# Simple Classification

This notebook builds a logistic regression model for each CUAD category (within the final 10). It follows the procedure outline below:

1. Loop through each spanned annotations and try to reverse find the source text that contains this annotation (positive chunk) - classify if a chunk contains a span
2. Find chunks that do not have annotations (negative chunks)
3. Turn source text to vectors using Roberta
4. Train classifier

# Defined Constants


In [1]:
from pathlib import Path

MODEL_ID = "distilroberta-base"

FROZEN_DIR = Path("task12b_outputs")
train_chunk_path = FROZEN_DIR / "train_chunks.parquet"
test_chunk_path = FROZEN_DIR / "test_chunks.parquet"
train_embedding_path = FROZEN_DIR / "train_embeddings.npz"
test_embedding_path = FROZEN_DIR / "test_embeddings.npz"

FINAL_IDS = [
    "governing_law",
    "renewal_term",
    "revenue_profit_sharing",
    "cap_on_liability",
    "uncapped_liability",
    "termination_for_convenience",
    "anti_assignment",
    "audit_rights",
    "license_grant",
    "exclusivity",
]

Load the CUAD dataset for the final categories.

In [2]:
from utils import load_file_from_github, load_split_data
training_data = load_split_data("train", cuad=True)

raw_test = load_file_from_github("data/cuad/test.json")
test_data = load_split_data("validation", cuad=True)

def get_final_10(data):
    # get all annotation sets in the final 10 categories
    sets = data.annotation_sets[data.annotation_sets.category_id.isin(FINAL_IDS)]

    # get all documents corresponding to the final annotation sets
    documents = data.documents[data.documents["contract_id"].isin(sets["contract_id"])]

    # get all spans in the final 10 categories
    spans = data.spans.merge(sets[["annotation_set_id", "category_id", "contract_id"]], on="annotation_set_id")

    return sets, documents, spans

train_sets, train_documents, train_spans = get_final_10(training_data)
test_sets, test_documents, test_spans = get_final_10(test_data)

train_spans.head()

,span_id,annotation_set_id,source_qa_id,answer_text,answer_start,answer_end,category_id,contract_id
0,contract_0001__renewal_term__span_000,contract_0001__renewal_term,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,If Distributor comp...,5543,5881,renewal_term,contract_0001
1,contract_0001__governing_law__span_000,contract_0001__governing_law,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,This Agreement is to be construed according to...,52061,52151,governing_law,contract_0001
2,contract_0001__exclusivity__span_000,contract_0001__exclusivity,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,The Company appoints the Distributor as an ex...,1854,2004,exclusivity,contract_0001
3,contract_0001__exclusivity__span_001,contract_0001__exclusivity,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Company hereby appoints ...,2112,2611,exclusivity,contract_0001
4,contract_0001__exclusivity__span_002,contract_0001__exclusivity,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,The Distributor shall not order or ...,12390,12503,exclusivity,contract_0001


# Positives

Loop through each spanned annotations and try to reverse find the source text that contains this annotation (positive chunk) - classify if a chunk contains a span

In [3]:
def find_positives(chunks, spans):
    """
    Reverse find the chunks contains an annotation (positive chunk)
    :param chunks: a DataFrame of chunks with at least chunk_id and text categories
    :return: a DataFrame of (chunk_id, category_id) pairs
    """
    chunks_by_contract_idx = chunks.groupby("contract_id").indices
    start, end = chunks.char_start.values, chunks.char_end.values

    hits, pos_found = [], 0
    for row in spans.itertuples():
        chunk_idxes = chunks_by_contract_idx.get(row.contract_id)

        # if contract has no annotation sets
        if chunk_idxes is None:
            continue

        # matches chunk indices where span start and end are contained in the chunk
        matches = chunk_idxes[(start[chunk_idxes] <= row.answer_start)
                              & (row.answer_end <= end[chunk_idxes])]
        pos_found += len(matches) > 0

        # saves the chunk_id and their category of each match
        hits += [(row.category_id, chunks.chunk_id.iat[i]) for i in matches]

    positives = pd.DataFrame(hits, columns=["category_id", "chunk_id"]).drop_duplicates()
    return positives, pos_found

# Negatives

Randomly find chunks that do not have annotations (negative chunks)

In [4]:
def get_chunk_categories(chunks, positives):
    """
    Pairs chunk_ids with each category it aligns with
    :param chunks: a DataFrame of chunks with at least chunk_id and text categories
    :param positives: a DataFrame of positive chunks with at least chunk_id and text categories
    :return: a DataFrame of (chunk_id, category_id) pairs
    """
    category_ids = positives["category_id"].unique().tolist()

    # cartesian product of chunk_id and category_id
    chunk_categories = pd.merge(chunks["chunk_id"], pd.Series(category_ids, name="category_id"), how="cross")

    # mark all tuples in positives as chunk contains category
    chunk_categories = chunk_categories.merge(
        positives.assign(is_contained=True),
        on=["chunk_id", "category_id"], how="left",
    )
    chunk_categories["is_contained"] = chunk_categories["is_contained"].fillna(False).astype(bool)

    return chunk_categories

# negatives are all tuples where is_contained == False
def find_negatives(chunk_categories):
    return chunk_categories.loc[~chunk_categories["is_contained"], ["chunk_id", "category_id"]]

# Building Chunks, Positives, and Negatives

This section builds out the chunks, positives, negatives, and chunk categories for the training data, using functions defined above. Chunks for training data are based on the approved frozen chunks.

**Chunking Strategy:**
- Chunking by paragraph
- DistilRoBERTa tokenizer
- Chunk Size = 512 tokens
- Overlap = 64 tokens
- Result: 10,841 Chunks

In [5]:
import pandas as pd

def load_or_build_chunks(cache_path, documents, raw_chunks=None, tokenizer=None,
                         chunk_size=512, max_tokens=512):
    """
    Loads chunks from a cache or builds new chunks. Chunks can be built from frozen or completely generated new.
    :param cache_path: Path location for the cached file and/or where to cache the new chunks
    :param documents: DataFrame of CUAD documents
    :param raw_chunks: DataFrame or None for the approved frozen chunks (typically from task10_outputs)
    :param tokenizer: Tokenizer or None for generating new chunks
    :param chunk_size: int representing the size of each generated chunk
    :param max_tokens: int representing the max number of chunks for the embedding model
    :return:
    """
    cache_path = Path(cache_path)
    if cache_path.exists():
        print("Retrieved data from cache.")
        return pd.read_parquet(cache_path)

    # if there's no cached chunks, use the approved chunks or generate new
    if raw_chunks is not None:
        chunks = pd.read_parquet(raw_chunks) if isinstance(raw_chunks, (str, Path)) else raw_chunks.copy()
    elif tokenizer is not None:
        chunks = create_paragraph_chunks(documents, tokenizer, chunk_size=chunk_size)
    else:
        raise ValueError("Provide cache_path (existing), raw_chunks, or tokenizer")

    # confirm that the max number of tokens does not exceed the embedding model's token limit
    if "num_tokens" in chunks.columns:
        assert chunks["num_tokens"].max() <= max_tokens, "chunk exceeds model max length"

    # confirm that all chunks belong to a document
    chunks = chunks.merge(
        documents[["contract_id", "context"]], on="contract_id", how="left", validate="many_to_one"
    )
    assert chunks["context"].notna().all(), "chunks reference contracts missing from documents"

    # append text to chunkd
    chunks["text"] = [
        ctx[s:e] for ctx, s, e in zip(chunks["context"], chunks["char_start"], chunks["char_end"])
    ]
    chunks = chunks.drop(columns="context")

    assert (chunks["text"].str.len() == chunks["char_end"] - chunks["char_start"]).all(), \
        "offsets run past the end of the document text"

    chunks.to_parquet(cache_path)
    return chunks

raw_train_chunks = load_file_from_github("notebooks/task10_outputs/chunks.parquet")
train_chunks = load_or_build_chunks(train_chunk_path, train_documents, raw_chunks=raw_train_chunks)
train_chunks.head()

Retrieved data from cache.


,method,contract_id,context_group_id,chunk_id,chunk_number,char_start,char_end,num_tokens,chunk_text,chunk_hash,document_hash,text
0,paragraph_based,contract_0001,context_41f7921a65,contract_0001__chunk_0000,0,0,2056,463,EXHIBIT 10.6\n\n ...,8d9734236056eaa8a3619bf2942ca9b27adadb920c52ff...,95ea6fae5227f2424896bb354e9de452c7f05a85dc0acb...,EXHIBIT 10.6\n\n ...
1,paragraph_based,contract_0001,context_41f7921a65,contract_0001__chunk_0001,1,2076,3126,512,1.1 Grant and Acceptance. Company h...,0eb8801fa97ea183343a724bb6fbd0caab779ccfcdb942...,95ea6fae5227f2424896bb354e9de452c7f05a85dc0acb...,1.1 Grant and Acceptance. Company h...
2,paragraph_based,contract_0001,context_41f7921a65,contract_0001__chunk_0002,2,2979,3717,359,such customers in the Market to those custom...,4e070f16a388ecff6151b6344f9ff838669ee88d9e2424...,95ea6fae5227f2424896bb354e9de452c7f05a85dc0acb...,such customers in the Market to those custom...
3,paragraph_based,contract_0001,context_41f7921a65,contract_0001__chunk_0003,3,3737,4788,512,1.2 License. The Company hereby grants ...,fe8b1735b9cc0a5536b6771e0f47e30c7c80c93992179d...,95ea6fae5227f2424896bb354e9de452c7f05a85dc0acb...,1.2 License. The Company hereby grants ...
4,paragraph_based,contract_0001,context_41f7921a65,contract_0001__chunk_0004,4,4658,5228,277,sublicense the Names or to do ...,c2b10a8c277aef96161b58f42a2068a32e017068e12f31...,95ea6fae5227f2424896bb354e9de452c7f05a85dc0acb...,sublicense the Names or to do ...


In [6]:
train_positives, pos_found = find_positives(train_chunks, train_spans)

print(f"{pos_found}/{len(train_spans)} spans matched a chunk")
train_positives.groupby("category_id").size().reset_index(name="count")

2990/3078 spans matched a chunk


,category_id,count
0,anti_assignment,313
1,audit_rights,259
2,cap_on_liability,333
3,exclusivity,203
4,governing_law,291
5,license_grant,386
6,renewal_term,127
7,revenue_profit_sharing,178
8,termination_for_convenience,143
9,uncapped_liability,91


In [7]:
train_chunk_categories = get_chunk_categories(train_chunks, train_positives)
train_negatives = find_negatives(train_chunk_categories)

train_negatives.groupby("category_id").size().reset_index(name="count")

,category_id,count
0,anti_assignment,10528
1,audit_rights,10582
2,cap_on_liability,10508
3,exclusivity,10638
4,governing_law,10550
5,license_grant,10455
6,renewal_term,10714
7,revenue_profit_sharing,10663
8,termination_for_convenience,10698
9,uncapped_liability,10750


In [9]:
train_chunk_categories.to_csv(FROZEN_DIR / "train_chunk_categories.csv", index=False)
print("Exported train_chunk_categories")

Exported train_chunk_categories


Confirm that no information was lost during the positive/negative computations and split.

In [8]:
assert len(train_chunk_categories) == len(train_chunks) * len(train_positives["category_id"].unique().tolist())
assert train_chunk_categories["is_contained"].sum() == len(train_positives)
assert len(train_negatives) + len(train_positives) == len(train_chunk_categories)

print("Everything adds up!")

Everything adds up!


# Imbalance

In the training set, every one of the 10 categories is evaluated against the same 10,841 chunks, and all of them are heavily imbalanced. The positive rate ranges from 0.84% (91 positives) for the rarest category to 3.56% (386 positives) for the most common, with an average of about 2.1% per category. This means there are between 27 and 118 negative chunks for every positive one. Across all categories there are only about 2,300 positive labels, so a typical chunk carries 0.21 labels on average and most chunks are negative for every category. A model that ignores the imbalance can score high accuracy by predicting “no clause” everywhere, so evaluation should rely on per-category precision, recall, and F1, and training should account for the imbalance (for example through class weighting).

In [9]:
imbalance = (
    train_chunk_categories.groupby("category_id")["is_contained"]
    .agg(n_pos="sum", n_total="size")
    .assign(
        n_neg=lambda d: d.n_total - d.n_pos,
        pos_rate=lambda d: d.n_pos / d.n_total,
        neg_per_pos=lambda d: (d.n_total - d.n_pos) / d.n_pos,
    )
    .sort_values("pos_rate")
)
imbalance.round(4)

,n_pos,n_total,n_neg,pos_rate,neg_per_pos
category_id,,,,,
uncapped_liability,91,10841,10750,0.0084,118.1319
renewal_term,127,10841,10714,0.0117,84.3622
termination_for_convenience,143,10841,10698,0.0132,74.8112
revenue_profit_sharing,178,10841,10663,0.0164,59.9045
exclusivity,203,10841,10638,0.0187,52.4039
audit_rights,259,10841,10582,0.0239,40.8571
governing_law,291,10841,10550,0.0268,36.2543
anti_assignment,313,10841,10528,0.0289,33.6358
cap_on_liability,333,10841,10508,0.0307,31.5556


In [10]:
imbalance["pos_rate"].sum()

np.float64(0.21437136795498574)

# Check GPU
If a powerful GPU is not locally available, it is not recommended to run the embedding cells. Use Google Colab's T4 GPU.

In [11]:
import torch
cuda_available = torch.cuda.is_available()
print("CUDA available:", cuda_available, "|", torch.cuda.get_device_name(0) if cuda_available else "select a GPU runtime")

CUDA available: False | select a GPU runtime


# Vector Embeddings
Turn source text to vectors using DistilRoBERTa. These vectors will be used as "features" when training the logistic regression model.

In [12]:
from sentence_transformers import SentenceTransformer
import numpy as np

def create_embeddings(chunks, prefix):
    model = SentenceTransformer("sentence-transformers/all-distilroberta-v1")

    chunks = chunks.reset_index(drop=True)      # row position == matrix position
    texts = chunks["text"].tolist()

    embeddings = model.encode(
        texts,
        batch_size=32,
        show_progress_bar=True,
        normalize_embeddings=True,   # unit-length, so dot product = cosine sim
    )                                # float32 numpy array, shape (len(texts), 768)

    assert embeddings.shape == (len(chunks), model.get_embedding_dimension())

    np.savez(f"{FROZEN_DIR}/{prefix}_embeddings.npz",
             chunk_ids=chunks["chunk_id"].to_numpy().astype(str),   # dtype '<U..', not object
             embeddings=embeddings)

def load_embeddings(prefix, chunks):
    d = np.load(f"{FROZEN_DIR}/{prefix}_embeddings.npz")
    embeddings, ids = d["embeddings"], d["chunk_ids"]

    chunks = chunks.reset_index(drop=True)

    # confirm that embeddings rows match the chunk_id rows
    assert len(embeddings) == len(chunks)
    assert (ids == chunks["chunk_id"].to_numpy().astype(str)).all()

    chunks["embedding"] = list(embeddings)
    return chunks, embeddings

/Users/amiebinan/Desktop/Internships/Accenture-1O-contract-review-challenge/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Compute or Load Vector Embeddings

In [13]:
if cuda_available and not train_embedding_path.exists():
    create_embeddings(train_chunks, "train")

train_chunks, train_embeddings = load_embeddings("train", train_chunks)
train_chunks.head()

,method,contract_id,context_group_id,chunk_id,chunk_number,char_start,char_end,num_tokens,chunk_text,chunk_hash,document_hash,text,embedding
0,paragraph_based,contract_0001,context_41f7921a65,contract_0001__chunk_0000,0,0,2056,463,EXHIBIT 10.6\n\n ...,8d9734236056eaa8a3619bf2942ca9b27adadb920c52ff...,95ea6fae5227f2424896bb354e9de452c7f05a85dc0acb...,EXHIBIT 10.6\n\n ...,"[-0.0101947775, -0.004966498, -0.0014591962, -..."
1,paragraph_based,contract_0001,context_41f7921a65,contract_0001__chunk_0001,1,2076,3126,512,1.1 Grant and Acceptance. Company h...,0eb8801fa97ea183343a724bb6fbd0caab779ccfcdb942...,95ea6fae5227f2424896bb354e9de452c7f05a85dc0acb...,1.1 Grant and Acceptance. Company h...,"[0.010386001, 0.00900911, 0.0069864546, -0.006..."
2,paragraph_based,contract_0001,context_41f7921a65,contract_0001__chunk_0002,2,2979,3717,359,such customers in the Market to those custom...,4e070f16a388ecff6151b6344f9ff838669ee88d9e2424...,95ea6fae5227f2424896bb354e9de452c7f05a85dc0acb...,such customers in the Market to those custom...,"[-0.011106098, -0.013900779, 0.015085381, 0.01..."
3,paragraph_based,contract_0001,context_41f7921a65,contract_0001__chunk_0003,3,3737,4788,512,1.2 License. The Company hereby grants ...,fe8b1735b9cc0a5536b6771e0f47e30c7c80c93992179d...,95ea6fae5227f2424896bb354e9de452c7f05a85dc0acb...,1.2 License. The Company hereby grants ...,"[-0.022413723, 0.049107153, -0.018923167, -0.0..."
4,paragraph_based,contract_0001,context_41f7921a65,contract_0001__chunk_0004,4,4658,5228,277,sublicense the Names or to do ...,c2b10a8c277aef96161b58f42a2068a32e017068e12f31...,95ea6fae5227f2424896bb354e9de452c7f05a85dc0acb...,sublicense the Names or to do ...,"[-0.007902839, 0.03549104, 0.008908994, -0.003..."


# Testing

We are using an 80/20 split of training and validation data (renamed "test" for convention).

In [14]:
from chunking_helper import create_paragraph_chunks
from transformers import AutoTokenizer

test_chunks = load_or_build_chunks(test_chunk_path, test_documents, tokenizer=AutoTokenizer.from_pretrained(MODEL_ID))

test_positives, _ = find_positives(test_chunks, test_spans)
test_chunk_categories = get_chunk_categories(test_chunks, test_positives)
test_negatives = find_negatives(test_chunk_categories)

if cuda_available and not test_embedding_path.exists():
    create_embeddings(test_chunks, "test")

test_chunks, test_embeddings = load_embeddings("test", test_chunks)
print("Test chunks and embeddings loaded successfully!")

Retrieved data from cache.
Test chunks and embeddings loaded successfully!


# Logistic Regression

A classifier for all provided categories by instantiating an individual LogisticRegression model for each category. It mimics the scikit-learn class when implementing hyperparameters, fitting, and prediction.

| Model         | Recall | F1 Score |
|---------------|--------|----------|
| Before tuning | 86.2%  | 31.8%    |
| After tuning  | 58.2%  | 53.5%    |


```python
# logistic_regression_by_category.py

from sklearn.linear_model import LogisticRegression
import pandas as pd

class LogisticRegressionByCategory:
    """One binary logistic regression per category, on chunk embeddings."""

    def __init__(self, categories, **lr_kwargs):
        """
        :param categories: ids of all categories to be classified
        :param lr_kwargs:  arguments for scikit-learn's logistic regression
        """
        self.lr_kwargs = {**lr_kwargs}
        self.categories = categories
        self.models = {}

    def fit(self, X, y):
        """
        Trains a Logistic Regression model for each of the categories
        :param X: np.array containing chunk embeddings for training data
        :param y: DataFrame aligned with X, providing 0/1 per category
        :return:
        """
        ...

    def predict_proba(self, X):
        """
        Predicts probability of a chunk being a part of a category, for every category
        :param X: a np.array containing chunk embeddings for test data
        :return:
        """
        ...

    def predict(self, X, threshold=0.5, index=None):
        """
        :param X: np.array of chunk embeddings
        :param threshold: float, or dict of category -> float
        :param index: chunk ids for the rows of X (e.g. test_chunks["chunk_id"])
        :return: DataFrame of 0/1 predictions, one column per category
        """
        ...
```


In [20]:
from logistic_regression_by_category import LogisticRegressionByCategory

X_train = train_embeddings
y_train = (train_chunk_categories
           .pivot(index="chunk_id", columns="category_id", values="is_contained")
           .reindex(train_chunks["chunk_id"])
           .fillna(0)
           .astype(int)
           )

X_test = test_embeddings
y_test = (test_chunk_categories
           .pivot(index="chunk_id", columns="category_id", values="is_contained")
           .reindex(test_chunks["chunk_id"])
           .fillna(0)
           .astype(int)
           )

clf = LogisticRegressionByCategory(FINAL_IDS, class_weight="balanced", random_state=42)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

y_pred

,governing_law,renewal_term,revenue_profit_sharing,cap_on_liability,uncapped_liability,termination_for_convenience,anti_assignment,audit_rights,license_grant,exclusivity
0,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,1,0,0
4,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...
2868,0,0,0,0,0,0,0,0,0,0
2869,0,0,0,0,0,0,0,0,0,0
2870,0,0,0,0,0,0,0,0,0,0
2871,1,0,0,0,0,0,1,0,0,0


In [21]:
from sklearn.metrics import precision_score, recall_score, f1_score

def evaluate(y_true, y_pred):
    """Both DataFrames: same index/columns, 0/1 values. Returns per-category metrics."""
    rows = []
    for cat in y_true.columns:
        t, p = y_true[cat].to_numpy(), y_pred[cat].to_numpy()
        rows.append({
            "category": cat,
            "support": t.sum(),
            "precision": precision_score(t, p, zero_division=0),
            "recall": recall_score(t, p, zero_division=0),
            "f1": f1_score(t, p, zero_division=0),
        })
    return pd.DataFrame(rows).set_index("category")

results = evaluate(y_test[FINAL_IDS], y_pred)
results[results["support"] > 0][["recall", "f1"]].mean()   # macro, ignoring empty categories

recall    0.861508
f1        0.317851
dtype: float64

# Hyperparameter Tuning

After running essentially a grid search over hyperparameters C and class_weight, and prediction threshold, the best combination is the following:

- `C`: 10; the inverse of regularization strength. A small C penalizes large weights heavily and keeps the model simple, and a large C lets it fit the training data more closely.
- `class_weight`: None; how much the loss counts mistakes on each class. "balanced" upweights the rare positive examples, and None treats every chunk equally.
- `threshold`: 0.20; the probability cutoff for calling a prediction positive. It’s applied after training and trades precision against recall.

The best results achieved are the following:

- Precision (`P`): 0.505; of the chunks the model flagged as containing a clause, the fraction that really do.
- Recall (`R`): of the chunks that really contain a clause, the fraction the model found.
`F1`: 0.535; the harmonic mean of precision and recall, which is high only when both are high.
- `PR-AUC`: 0.529; the area under the precision-recall curve across all thresholds, measuring how well the model ranks true clauses above non-clauses, independent of any cutoff.

In [28]:
from itertools import product
from sklearn.metrics import average_precision_score, precision_recall_fscore_support

supported = y_test.columns[y_test.sum() > 0]
yt = y_test[supported].to_numpy()

best = {"model": None, "prob": None, "score": 0, "C": 0, "cw": 0}
for C, cw in product([0.1, 1, 10, 100], [None, "balanced"]):
    m = LogisticRegressionByCategory(FINAL_IDS, random_state=42, C=C, class_weight=cw, max_iter=1000)
    m.fit(X_train, y_train)
    prob = m.predict_proba(X_test, index=test_chunks["chunk_id"])[supported].to_numpy()
    score = average_precision_score(yt, prob, average="macro")
    print(f"C={C}, class_weight={cw}, PR-AUC={score:.3f}")
    if best is None or score > best["score"]:
        best = {"model": m, "prob": prob, "score": score, "C": C, "cw": cw}

def prf(t):
    return precision_recall_fscore_support(
        yt, (best["prob"] >= t).astype(int), average="macro", zero_division=0
    )[:3]

thresholds = np.linspace(0.01, 0.99, 99)
best_t = max(thresholds, key=lambda t: prf(t)[2])
best["model"].threshold_ = float(best_t)

p, r, f = prf(best_t)
print(f"\nC={best['C']}, class_weight={best['cw']}, threshold={best_t:.2f}")
print(f"P={p:.3f}  R={r:.3f}  F1={f:.3f}  PR-AUC={best['score']:.3f}")

C=0.1, class_weight=None, PR-AUC=0.371
C=0.1, class_weight=balanced, PR-AUC=0.443
C=1, class_weight=None, PR-AUC=0.471
C=1, class_weight=balanced, PR-AUC=0.501
C=10, class_weight=None, PR-AUC=0.529
C=10, class_weight=balanced, PR-AUC=0.512
C=100, class_weight=None, PR-AUC=0.519
C=100, class_weight=balanced, PR-AUC=0.490

C=10, class_weight=None, threshold=0.20
P=0.505  R=0.582  F1=0.535  PR-AUC=0.529


Freeze the final model.

In [27]:
import pickle

with open(FROZEN_DIR / "logistic_baseline.pkl", "wb") as file:
    pickle.dump(best["model"], file)

print("Model saved successfully!")

Model saved successfully!
